In [ ]:
from google.colab import files
uploaded = files.upload()
print("Yüklenen dosyalar:", list(uploaded.keys()))

Saving HOG.zip to HOG.zip
Yüklenen dosyalar: ['HOG.zip']


In [ ]:
import os, zipfile, shutil

# repo klasörünü temizliyoruz
repo_dir = "workspace/repo"
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)
os.makedirs(repo_dir, exist_ok=True)

# Colab'a yüklenen zip'i bul
zip_files = [f for f in os.listdir(".") if f.lower().endswith(".zip")]
if not zip_files:
    raise FileNotFoundError("Zip bulunamadı. Yükleme yapılmamış olabilir.")

zip_path = zip_files[0]
print("Bulunan zip:", zip_path)

# zip'i repo klasörüne aç
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(repo_dir)

print("Zip açıldı ", repo_dir)

Bulunan zip: HOG.zip
Zip açıldı  workspace/repo


In [ ]:
!ls -la workspace/repo


total 12
drwxr-xr-x 3 root root 4096 Dec 29 04:26 .
drwxr-xr-x 3 root root 4096 Dec 29 04:26 ..
drwxr-xr-x 8 root root 4096 Dec 29 04:26 HOG


In [ ]:
import os, shutil

repo_dir = "workspace/repo"
items = os.listdir(repo_dir)

# Eğer repo_dir içinde tek bir klasör varsa ve asıl dosyalar onun içindeyse, onu yukarı taşı
if len(items) == 1 and os.path.isdir(os.path.join(repo_dir, items[0])):
    inner = os.path.join(repo_dir, items[0])
    print("İç içe klasör bulundu:", items[0], "→ köke taşınıyor...")

    for name in os.listdir(inner):
        shutil.move(os.path.join(inner, name), os.path.join(repo_dir, name))

    shutil.rmtree(inner)
    print("Taşıma tamamlandı.")
else:
    print("Repo zaten kökte görünüyor (iç içe klasör yok).")

!ls -la workspace/repo | head -n 50


İç içe klasör bulundu: HOG → köke taşınıyor...
Taşıma tamamlandı.
total 3060
drwxr-xr-x 8 root root    4096 Dec 29 04:26 .
drwxr-xr-x 3 root root    4096 Dec 29 04:26 ..
drwxr-xr-x 2 root root    4096 Dec 29 04:26 assest
drwxr-xr-x 2 root root    4096 Dec 29 04:26 assets
-rw-r--r-- 1 root root    1057 Dec 29 04:26 base_classes.py
-rw-r--r-- 1 root root    1954 Dec 29 04:26 config.py
drwxr-xr-x 4 root root    4096 Dec 29 04:26 data
-rw-r--r-- 1 root root    1778 Dec 29 04:26 enhancements.py
-rw-r--r-- 1 root root     868 Dec 29 04:26 exceptions.py
-rw-r--r-- 1 root root    3698 Dec 29 04:26 factories.py
-rw-r--r-- 1 root root   16033 Dec 29 04:26 filters.py
drwxr-xr-x 7 root root    4096 Dec 29 04:26 .git
-rw-r--r-- 1 root root      79 Dec 29 04:26 .gitignore
-rw-r--r-- 1 root root    3371 Dec 29 04:26 gui_components_empty.py
-rw-r--r-- 1 root root    6947 Dec 29 04:26 gui_components.py
-rw-r--r-- 1 root root    7403 Dec 29 04:26 hog_implementation.py
-rw-r--r-- 1 root root 2941778 Dec 

In [ ]:
import os, json, hashlib, time

REPO_ROOT = "workspace/repo"
OUT_INDEX = "outputs/context/index.json"

WHITELIST_EXT = {".py", ".md", ".txt", ".yml", ".yaml", ".json"}

BLACKLIST_DIRS = {
    ".git", ".venv", "venv", "__pycache__", ".mypy_cache", ".pytest_cache",
    "node_modules", "dist", "build", ".next", ".idea", ".vscode"
}

MAX_FILE_BYTES = 1_500_000  # 1.5MB üstü dosyaları MVP'de atlıyoruz

def sha1_text(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()

def is_allowed_file(path: str) -> bool:
    ext = os.path.splitext(path.lower())[1]
    if ext not in WHITELIST_EXT:
        return False
    try:
        return os.path.getsize(path) <= MAX_FILE_BYTES
    except:
        return False

manifest = {
    "repo_root": REPO_ROOT,
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "whitelist_ext": sorted(list(WHITELIST_EXT)),
    "blacklist_dirs": sorted(list(BLACKLIST_DIRS)),
    "files": []
}

for root, dirs, files in os.walk(REPO_ROOT):
    # kara listedeki klasörlere girme
    dirs[:] = [d for d in dirs if d not in BLACKLIST_DIRS]

    for fn in files:
        full = os.path.join(root, fn)
        rel = os.path.relpath(full, REPO_ROOT).replace("\\", "/")

        if not is_allowed_file(full):
            continue

        try:
            with open(full, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()
        except:
            continue

        manifest["files"].append({
            "path": rel,
            "bytes": os.path.getsize(full),
            "sha1": sha1_text(text),
        })

os.makedirs(os.path.dirname(OUT_INDEX), exist_ok=True)
with open(OUT_INDEX, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("index.json yazıldı:", OUT_INDEX)
print("Dahil edilen dosya sayısı:", len(manifest["files"]))
print("Örnek ilk 10 dosya:")
for x in manifest["files"][:10]:
    print("-", x["path"])


index.json yazıldı: outputs/context/index.json
Dahil edilen dosya sayısı: 19
Örnek ilk 10 dosya:
- image_manager.py
- exceptions.py
- gui_components_empty.py
- factories.py
- logger.py
- README.md
- enhancements.py
- gui_components.py
- config.py
- base_classes.py


In [ ]:
"""
chunking -- dosyaları parçalara ayırma işlemi -- ki retrieval(arama/geri getirme) düzügn çalışsın

.py -> function/class bazlı chunk
.md -> başlık bazlı chunk

çıktı --> outputs/context/chunks.jsonl (her satır 1 chunk)

"""

'\nchunking -- dosyaları parçalara ayırma işlemi -- ki retrieval(arama/geri getirme) düzügn çalışsın\n\n.py -> function/class bazlı chunk\n.md -> başlık bazlı chunk\n\nçıktı --> outputs/context/chunks.jsonl (her satır 1 chunk)\n\n'

In [ ]:

import os, json, re

INDEX_PATH = "outputs/context/index.json"
REPO_ROOT = "workspace/repo"
OUT_CHUNKS = "outputs/context/chunks.jsonl"

def read_text(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def chunk_python(text):
    """
    Basit MVP chunking:
    - class ...: veya def ...: satırlarından itibaren bir sonraki class/def'e kadar
    """
    lines = text.splitlines()
    starts = []
    pattern = re.compile(r"^\s*(def|class)\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(")
    pattern_class = re.compile(r"^\s*class\s+([A-Za-z_][A-Za-z0-9_]*)\s*[\(:]")
    for i, ln in enumerate(lines):
        if pattern.search(ln) or pattern_class.search(ln):
            starts.append(i)

    # hiç def/class yoksa tüm dosyayı tek chunk yap
    if not starts:
        return [("file", 0, len(lines), "\n".join(lines))]

    chunks = []
    for idx, s in enumerate(starts):
        e = starts[idx+1] if idx+1 < len(starts) else len(lines)
        chunk_text = "\n".join(lines[s:e]).strip()
        if chunk_text:
            # başlık çıkar
            m1 = pattern.search(lines[s])
            m2 = pattern_class.search(lines[s])
            if m1:
                kind, name = m1.group(1), m1.group(2)
                title = f"{kind} {name}"
            elif m2:
                title = f"class {m2.group(1)}"
            else:
                title = "block"
            chunks.append((title, s+1, e, chunk_text))
    return chunks

def chunk_markdown(text):
    """
    Markdown başlık bazlı:
    - #, ##, ### ile başlayan satırlar section başlangıcı.
    """
    lines = text.splitlines()
    header_idx = [i for i, ln in enumerate(lines) if re.match(r"^\s{0,3}#{1,6}\s+\S+", ln)]
    if not header_idx:
        return [("md_file", 0, len(lines), "\n".join(lines))]

    chunks = []
    for j, s in enumerate(header_idx):
        e = header_idx[j+1] if j+1 < len(header_idx) else len(lines)
        title = lines[s].strip()
        chunk_text = "\n".join(lines[s:e]).strip()
        if chunk_text:
            chunks.append((title, s+1, e, chunk_text))
    return chunks

# index oku
with open(INDEX_PATH, "r", encoding="utf-8") as f:
    index = json.load(f)

all_chunks = []
chunk_counter = 0

for fileinfo in index["files"]:
    rel = fileinfo["path"]
    full = os.path.join(REPO_ROOT, rel)
    ext = os.path.splitext(rel.lower())[1]
    text = read_text(full)

    if ext == ".py":
        pieces = chunk_python(text)
    elif ext == ".md":
        pieces = chunk_markdown(text)
    else:
        # txt/yaml/json: dosya bazlı tek chunk
        pieces = [("file", 0, len(text.splitlines()), text)]

    for title, start_line, end_line, chunk_text in pieces:
        chunk_id = f"chunk_{chunk_counter:05d}"
        all_chunks.append({
            "chunk_id": chunk_id,
            "source_path": rel,
            "title": title,
            "start_line": start_line,
            "end_line": end_line,
            "text": chunk_text
        })
        chunk_counter += 1

# jsonl yaz
os.makedirs(os.path.dirname(OUT_CHUNKS), exist_ok=True)
with open(OUT_CHUNKS, "w", encoding="utf-8") as f:
    for ch in all_chunks:
        f.write(json.dumps(ch, ensure_ascii=False) + "\n")

print("chunks.jsonl yazıldı:", OUT_CHUNKS)
print("Toplam chunk sayısı:", len(all_chunks))

# hızlı örnek göster
print("\nİlk 3 chunk örneği:")
for ch in all_chunks[:3]:
    print("-", ch["chunk_id"], ch["source_path"], "|", ch["title"], f"({ch['start_line']}-{ch['end_line']})")

chunks.jsonl yazıldı: outputs/context/chunks.jsonl
Toplam chunk sayısı: 194

İlk 3 chunk örneği:
- chunk_00000 image_manager.py | class ImageManager (8-10)
- chunk_00001 image_manager.py | def __init__ (11-16)
- chunk_00002 image_manager.py | def load_image (17-27)


In [ ]:
!pip -q install chromadb sentence-transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.8 MB/s eta 

In [ ]:
!pip -q install llama-cpp-python==0.3.7 huggingface_hub
print("llama-cpp-python + huggingface_hub kuruldu")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.7/66.7 MB 5.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.3 MB/s eta 0:00:00
llama-cpp-python + huggingface_hub kuruldu


In [ ]:
from huggingface_hub import hf_hub_download
import os

REPO_ID = "AIronMind/Qwen2.5-Coder-3B-Instruct-Q4_K_M-GGUF"
FILENAME = "qwen2.5-coder-3b-instruct-q4_k_m.gguf"

local_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    local_dir="models",
    local_dir_use_symlinks=False
)

print("İndirildi:", local_path)
print("Boyut (GB):", round(os.path.getsize(local_path)/1024/1024/1024, 2))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


qwen2.5-coder-3b-instruct-q4_k_m.gguf:   0%|          | 0.00/1.93G [00:00<?, ?B/s]

İndirildi: models/qwen2.5-coder-3b-instruct-q4_k_m.gguf
Boyut (GB): 1.8


In [ ]:
from llama_cpp import Llama

model_path = "models/qwen2.5-coder-3b-instruct-q4_k_m.gguf"

llm = Llama(
    model_path=model_path,
    n_ctx=2048,
    n_threads=4,
    verbose=False,
)

messages = [
    {"role": "system", "content": "You are a helpful coding assistant. Answer in Turkish, concise and clear."},
    {"role": "user", "content": "Kısaca: Sen kimsin ve bu projede ne yapacaksın? (2-3 cümle)"},
]

resp = llm.create_chat_completion(
    messages=messages,
    temperature=0.2,       # daha az saçmalama
    top_p=0.9,
    max_tokens=120,
    repeat_penalty=1.15,   # tekrarları azaltır
)

print(resp["choices"][0]["message"]["content"])

llama_init_from_model: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Ben bir yazılım geliştiricisiyim ve bu projede, kullanıcıların ihtiyaçlarını karşılamak için uygulamalar geliştirmeye çalışıyorum.


In [ ]:
import shutil, os

CHROMA_DIR = "outputs/context/vector_db"
if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)
print("Eski vector_db silindi.")


Eski vector_db silindi.


In [ ]:
import json, re, math
from collections import Counter, defaultdict

CHUNKS_PATH = "outputs/context/chunks.jsonl"

def tokenize(s: str):
    s = s.lower()
    # kelimeleri al (kod + metin)
    return re.findall(r"[a-zA-Z_]\w+|[0-9]+", s)

# 1) chunkları yükle
chunks = []
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line))

print("Chunk yüklendi:", len(chunks))

# 2) Basit bir arama indeksi kur (TF-IDF benzeri)
doc_tokens = []
df = Counter()

for ch in chunks:
    toks = tokenize(ch["text"])
    doc_tokens.append(toks)
    for t in set(toks):
        df[t] += 1

N = len(chunks)
idf = {t: math.log((N + 1) / (df[t] + 1)) + 1.0 for t in df}  # smooth idf

def score(query: str, idx: int):
    q = tokenize(query)
    if not q:
        return 0.0
    tf = Counter(doc_tokens[idx])
    s = 0.0
    for t in q:
        if t in tf:
            s += (1 + math.log(tf[t])) * idf.get(t, 0.0)
    return s

def retrieve(query: str, k=5):
    scored = [(score(query, i), i) for i in range(N)]
    scored.sort(reverse=True, key=lambda x: x[0])
    top = [chunks[i] for (sc, i) in scored[:k] if sc > 0]
    return top

# 3) TEST: senin projen için mantıklı bir soru soralım
test_query = "Bu proje nasıl çalıştırılır? arayüz veya gui var mı?"
top = retrieve(test_query, k=5)

print("\nSoru:", test_query)
print("Bulunan Top-5 chunk:")
for j, ch in enumerate(top, 1):
    preview = ch["text"][:180].replace("\n", " ")
    print(f"{j}) {ch['chunk_id']} | {ch['source_path']} | {ch['title']} | {preview}...")

Chunk yüklendi: 194

Soru: Bu proje nasıl çalıştırılır? arayüz veya gui var mı?
Bulunan Top-5 chunk:
1) chunk_00183 | train_model.py | def load_data | def load_data():     data = []     labels = []          # 1. Pozitif (Nesne Var) Resimleri Yükle     print(f"Pozitif resimler okunuyor...")     if not os.path.exists(POS_PATH):    ...
2) chunk_00186 | utils.py | def is_valid_image_file | def is_valid_image_file(file_path: str) -> bool:         """Check if file is a valid image"""         #Check if file exists         #Check if file extension is in SUPPORTED_EXTENSI...
3) chunk_00091 | main_app_empty.py | def _setup_gui | def _setup_gui(self):         """Setup GUI components"""         # TODO: Create main frame with padding         # TODO: Configure grid layout         # TODO: Create title label    ...
4) chunk_00165 | hog_implementation.py | def visualize_hog_pil | def visualize_hog_pil(pil_image, cell_size=(12, 12), num_bins=10):     """     GUI'den gelen PIL.Image için HOG görselleştir

In [ ]:
import os, json, math
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# 1) chunks.jsonl'ı oku
# 2) Chroma (vector DB) oluştur
# 3) Her chunk'ı (metin parçası) DB'ye kaydet
#    -> metadata: source_path + start_line/end_line + title

CHROMA_DIR = "outputs/context/vector_db"
COLLECTION = "repo_chunks"
CHUNKS_PATH = "outputs/context/chunks.jsonl"

assert os.path.exists(CHUNKS_PATH), "chunks.jsonl yok! Chunking adımı çalışmamış."

# 1) Chunk'ları RAM'e yükle
chunks = []
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))

print("Chunk yüklendi:", len(chunks))

# 2) Embedding modeli (arama motoru gibi)
embed_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# 3) Chroma'yı başlat
client = chromadb.PersistentClient(path=CHROMA_DIR)

# 4) Koleksiyonu sıfırdan kur (eski kayıtlarla karışmasın)
try:
    client.delete_collection(COLLECTION)
except Exception:
    pass

col = client.create_collection(name=COLLECTION, embedding_function=embed_fn)

# 5) DB'ye ekle (batch ekliyoruz ki büyük repolarda da sorunsuz olsun)
ids = [c["chunk_id"] for c in chunks]
docs = [c.get("text", "") for c in chunks]
metas = [
    {
        "source_path": c.get("source_path"),
        "title": c.get("title"),
        "start_line": c.get("start_line"),
        "end_line": c.get("end_line"),
    }
    for c in chunks
]

BATCH = 500
total = len(ids)
for start in range(0, total, BATCH):
    end = min(start + BATCH, total)
    col.add(
        ids=ids[start:end],
        documents=docs[start:end],
        metadatas=metas[start:end],
    )
    if (start // BATCH) % 10 == 0:
        print(f"  eklendi: {end}/{total}")

print("Vector DB hazır. Kayıt sayısı:", col.count())


Chunk yüklendi: 194


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  eklendi: 194/194
Vector DB hazır. Kayıt sayısı: 194


In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

CHROMA_DIR = "outputs/context/vector_db"
COLLECTION = "repo_chunks"   # dikkat: repo_chunks, repo_chunks değil

embed_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path=CHROMA_DIR)

print("Mevcut collection'lar:", [c.name for c in client.list_collections()])

# VARSA açar, YOKSA oluşturur -> hata vermez
col = client.get_or_create_collection(name=COLLECTION, embedding_function=embed_fn)

print("Bağlandı:", col.name)


Mevcut collection'lar: ['repo_chunks']
Bağlandı: repo_chunks


In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

CHROMA_DIR = "outputs/context/vector_db"
COLLECTION = "repo_chunks"

embed_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path=CHROMA_DIR)

col = client.get_or_create_collection(name=COLLECTION, embedding_function=embed_fn)

query = "MenuManager sınıfı ne yapıyor?" #örnek
res = col.query(query_texts=[query], n_results=5)

print("=== Bulunan QA delilleri ===")
for i, doc in enumerate(res["documents"][0]):
    meta = res["metadatas"][0][i]
    source = meta.get("source_path", "bilinmiyor")
    start = meta.get("start_line", "?")
    end = meta.get("end_line", "?")
    title = meta.get("title", "")
    print(f"{i+1}) {source} | L{start}-L{end} | {title}")


=== Bulunan QA delilleri ===
1) main_app.py | L134-L138 | def _setup_menu
2) main_app_empty.py | L80-L85 | def _setup_menu
3) gui_components_empty.py | L9-L11 | class MenuManager
4) gui_components.py | L9-L11 | class MenuManager
5) filters.py | L179-L186 | def __init__


In [ ]:
# 1) Chroma'dan gelen top parçaları context'e çevir
contexts = []
for i, doc in enumerate(res["documents"][0]):
    meta = res["metadatas"][0][i]
    source = meta.get("source_path", "bilinmiyor")
    start = meta.get("start_line", "?")
    end = meta.get("end_line", "?")
    title = meta.get("title", "")
    short = doc[:800]

    contexts.append(
        f"KANIT #{i+1}\n"
        f"Dosya: {source}\n"
        f"Satır: L{start}-L{end}\n"
        f"Başlık: {title}\n"
        f"Alıntı:\n{short}"
    )

context_text = "\n\n---\n\n".join(contexts)
print(context_text[:1000])  # sadece kontrol amaçlı ilk kısmı yazdır


KANIT #1
Dosya: main_app.py
Satır: L134-L138
Başlık: def _setup_menu
Alıntı:
def _setup_menu(self):
        """Setup menu"""
        self.menu_manager = MenuManager(self.root, self)
        self.menu_manager.create_menu()

---

KANIT #2
Dosya: main_app_empty.py
Satır: L80-L85
Başlık: def _setup_menu
Alıntı:
def _setup_menu(self):
        """Setup menu"""
        # TODO: Create MenuManager instance
        # TODO: Create menu
        pass

---

KANIT #3
Dosya: gui_components_empty.py
Satır: L9-L11
Başlık: class MenuManager
Alıntı:
class MenuManager:
    """Manages application menu bar"""

---

KANIT #4
Dosya: gui_components.py
Satır: L9-L11
Başlık: class MenuManager
Alıntı:
class MenuManager:
    """Manages application menu bar"""

---

KANIT #5
Dosya: filters.py
Satır: L179-L186
Başlık: def __init__
Alıntı:
def __init__(self):
        super().__init__("Person Detection")
        self.hog = cv2.HOGDescriptor() #opencv'nin hazır hog tanımlayıcısını başlat
        self.hog.setSVMDetector(

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path="models/qwen2.5-coder-3b-instruct-q4_k_m.gguf",
    n_ctx=2048,
    n_threads=4,
    verbose=False
)

query = "MenuManager sınıfı ne yapıyor ve menü bar nerede kuruluyor?"

messages = [
    {
        "role": "system",
        "content": (
            "Türkçe yaz.\n"
            "SADECE CONTEXT'teki KANIT'lara dayan.\n"
            "Eğer bir bilgi KANIT'larda yoksa 'BULAMADIM' de.\n"
            "Her iddia için en az 1 kanıt yazmak ZORUNDASIN.\n"
            "Kanıt formatı aynen şöyle olmalı (numara değil!):\n"
            "- Dosya: <dosya>\n"
            "  Satır: Lx-Ly\n"
            "  Alıntı: <CONTEXT'ten 1 satır>\n"
            "Asla '[1]' gibi referans yazma."
        )
    },
    {
        "role": "user",
        "content": (
            f"SORU:\n{query}\n\n"
            f"CONTEXT:\n{context_text}\n\n"
            "ÇIKTI FORMAT:\n"
            "1) Cevap (2-4 cümle)\n"
            "2) Kanıtlar (en az 2 madde)\n"
        )
    }
]

resp = llm.create_chat_completion(
    messages=messages,
    temperature=0.1,
    max_tokens=250
)

print(resp["choices"][0]["message"]["content"])


llama_init_from_model: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


1) MenuManager sınıfı, uygulamanın ana penceresine bir menü barı eklemek için kullanılır. Menü bar, uygulamanın ana penceresinin üstünde yer alır.

2) Kanıtlar:
   - Kanıt #1: main_app.py dosyasında _setup_menu fonksiyonu, MenuManager sınıfının root ve self parametrelerini alarak bir instance oluşturur ve menu barı oluşturur.
   - Kanıt #3: gui_components_empty.py dosyasında MenuManager sınıfı tanımlanmıştır ve bu sınıf, uygulamanın menü barını yönetmek için kullanılır.


In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

CHROMA_DIR = "outputs/context/vector_db"
COLLECTION = "repo_chunks"

embed_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path=CHROMA_DIR)
col = client.get_collection(name=COLLECTION, embedding_function=embed_fn)

query = "MenuManager sınıfı ne yapıyor?"
res = col.query(query_texts=[query], n_results=8)

print("Soru:", query)
print("\nVector DB Top-8 sonuç (GUI odaklı):")
for i in range(8):
    cid = res["ids"][0][i]
    meta = res["metadatas"][0][i]
    snippet = res["documents"][0][i][:140].replace("\n", " ")
    print(f"{i+1}) {cid} | {meta.get('source_path')} | {meta.get('title')} | {snippet}...")


Soru: MenuManager sınıfı ne yapıyor?

Vector DB Top-8 sonuç (GUI odaklı):
1) chunk_00174 | main_app.py | def _setup_menu | def _setup_menu(self):         """Setup menu"""         self.menu_manager = MenuManager(self.root, self)         self.menu_manager.create_me...
2) chunk_00096 | main_app_empty.py | def _setup_menu | def _setup_menu(self):         """Setup menu"""         # TODO: Create MenuManager instance         # TODO: Create menu         pass...
3) chunk_00015 | gui_components_empty.py | class MenuManager | class MenuManager:     """Manages application menu bar"""...
4) chunk_00061 | gui_components.py | class MenuManager | class MenuManager:     """Manages application menu bar"""...
5) chunk_00144 | filters.py | def __init__ | def __init__(self):         super().__init__("Person Detection")         self.hog = cv2.HOGDescriptor() #opencv'nin hazır hog tanımlayıcısın...
6) chunk_00063 | gui_components.py | def create_menu | def create_menu(self):         """Create menu bar"""    

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

CHROMA_DIR = "outputs/context/vector_db"
COLLECTION = "repo_chunks"

embed_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path=CHROMA_DIR)
col = client.get_or_create_collection(name=COLLECTION, embedding_function=embed_fn)

query = "MenuManager sınıfı ne yapıyor ve menü bar nerede kuruluyor?"
res = col.query(query_texts=[query], n_results=5)

# Kanıt listesi: modelin uyduramayacağı gerçek kaynaklar
EVIDENCE = []
for i, doc in enumerate(res["documents"][0], start=1):
    meta = res["metadatas"][0][i-1]
    ev = {
        "id": f"E{i}",
        "source_path": meta.get("source_path", "bilinmiyor"),
        "start_line": meta.get("start_line", "?"),
        "end_line": meta.get("end_line", "?"),
        "title": meta.get("title", ""),
        "quote": "\n".join(doc.splitlines()[:2])  # ilk 1-2 satır
    }
    EVIDENCE.append(ev)

print("=== KANITLAR (gerçek kaynak) ===")
for e in EVIDENCE:
    print(f"{e['id']} | {e['source_path']} | L{e['start_line']}-L{e['end_line']} | {e['title']}")
    print(f"ALINTI: {e['quote']}\n")


=== KANITLAR (gerçek kaynak) ===
E1 | gui_components_empty.py | L9-L11 | class MenuManager
ALINTI: class MenuManager:
    """Manages application menu bar"""

E2 | gui_components.py | L9-L11 | class MenuManager
ALINTI: class MenuManager:
    """Manages application menu bar"""

E3 | main_app.py | L134-L138 | def _setup_menu
ALINTI: def _setup_menu(self):
        """Setup menu"""

E4 | gui_components_empty.py | L17-L24 | def create_menu
ALINTI: def create_menu(self):
        """Create menu bar"""

E5 | gui_components.py | L17-L28 | def create_menu
ALINTI: def create_menu(self):
        """Create menu bar"""



In [ ]:
import json
from llama_cpp import Llama

llm = Llama(
    model_path="models/qwen2.5-coder-3b-instruct-q4_k_m.gguf",
    n_ctx=2048,
    n_threads=4,
    verbose=False
)

# LLM'e verilecek kısa kanıt özeti (dosya adı + satır + alıntı)
evidence_text = "\n".join(
    [f"{e['id']} | {e['source_path']} | L{e['start_line']}-L{e['end_line']}\n{e['quote']}\n"
     for e in EVIDENCE]
)

messages = [
    {
        "role": "system",
        "content": (
            "Türkçe yaz.\n"
            "SADECE aşağıdaki KANITLARA dayan.\n"
            "Dosya adı veya satır numarası UYDURMA.\n"
            "Sadece kanıt kimlikleri (E1, E2...) seçebilirsin.\n"
            "JSON dışında çıktı verme."
        )
    },
    {
        "role": "user",
        "content": (
            f"SORU: {query}\n\n"
            f"KANITLAR:\n{evidence_text}\n\n"
            "ŞU JSON'U üret:\n"
            "{\n"
            '  "answer": "2-4 cümlelik cevap",\n'
            '  "use_evidence": ["E1","E3"]\n'
            "}\n"
        )
    }
]

resp = llm.create_chat_completion(messages=messages, temperature=0.1, max_tokens=250)
raw = resp["choices"][0]["message"]["content"].strip()

# JSON parse (bazen model başına/sonuna boşluk koyabiliyor)
data = json.loads(raw)

print("=== CEVAP ===")
print(data["answer"])

print("\n=== KANITLAR ===")
used = set(data.get("use_evidence", []))
for e in EVIDENCE:
    if e["id"] in used:
        print(f"- {e['source_path']} | L{e['start_line']}-L{e['end_line']} | {e['title']}")
        print(f"  Alıntı: {e['quote']}")


llama_init_from_model: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


=== CEVAP ===
MenuManager sınıfı, uygulama menü barını yönetir. Menü bar, main_app.py dosyasındaki _setup_menu() metodunda kuruluyor.

=== KANITLAR ===
- gui_components_empty.py | L9-L11 | class MenuManager
  Alıntı: class MenuManager:
    """Manages application menu bar"""
- main_app.py | L134-L138 | def _setup_menu
  Alıntı: def _setup_menu(self):
        """Setup menu"""


In [ ]:
#dseneme
query = "MenuManager sınıfı ne yapıyor?"
res = col.query(query_texts=[query], n_results=5)

print("=== Bulunan parçalar (retrieval testi) ===")
for i, doc in enumerate(res["documents"][0]):
    meta = res["metadatas"][0][i]
    sp = meta.get("source_path", "bilinmiyor")
    s  = meta.get("start_line", "?")
    e  = meta.get("end_line", "?")
    t  = meta.get("title", "")
    print(f"{i+1}) {sp} | L{s}-L{e} | {t}\n{doc.splitlines()[0]}\n")


=== Bulunan parçalar (retrieval testi) ===
1) main_app.py | L134-L138 | def _setup_menu
def _setup_menu(self):

2) main_app_empty.py | L80-L85 | def _setup_menu
def _setup_menu(self):

3) gui_components_empty.py | L9-L11 | class MenuManager
class MenuManager:

4) gui_components.py | L9-L11 | class MenuManager
class MenuManager:

5) filters.py | L179-L186 | def __init__
def __init__(self):



In [ ]:
query = "MenuManager sınıfı ne yapıyor?"
res = col.query(query_texts=[query], n_results=1)

print(res["metadatas"][0][0])


{'source_path': 'main_app.py', 'start_line': 134, 'title': 'def _setup_menu', 'end_line': 138}


In [ ]:
from llama_cpp import Llama
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# 1) GUI odaklı retrieval
CHROMA_DIR = "outputs/context/vector_db"
COLLECTION = "repo_chunks"
embed_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path=CHROMA_DIR)
col = client.get_collection(name=COLLECTION, embedding_function=embed_fn)

# query = "Bu projede GUI var mı? Nereden çalıştırılır? tkinter main.py root.mainloop"
query = "MenuManager sınıfı ne yapıyor?"
res = col.query(query_texts=[query], n_results=5)

contexts = []
for i in range(5):
    meta = res["metadatas"][0][i]
    txt = res["documents"][0][i]
    # context'i kısalt: LLM'in odağını kaybetmesin
    short = txt[:900]
    contexts.append(f"[{i+1}] {meta.get('source_path')} | {meta.get('title')}\n{short}")

context_text = "\n\n".join(contexts)

# 2) Qwen
llm = Llama(
    model_path="models/qwen2.5-coder-3b-instruct-q4_k_m.gguf",
    n_ctx=2048,
    n_threads=4,
    verbose=False
)

messages = [
    {
        "role": "system",
        "content": (
            "Türkçe yaz. Kısa ve net ol.\n"
            "Sadece CONTEXT'e dayan.\n"
            "Eğer CONTEXT içinde açık bir işaret bulamazsan 'Bulamadım' de.\n"
            "Cevabında mutlaka 1-2 kısa kod satırını (snippet) alıntıla."
        )
    },
    {
        "role": "user",
        "content": (
            f"SORU: {query}\n\n"
            f"CONTEXT:\n{context_text}\n\n"
            "ÇIKTI FORMAT:\n"
             "1)MenuManager nedir?"
             "2)MenuManager hangi dosyada?"
             "3)Uygulamanın menü bar’ı nerede yönetiliyor?"

            # "1) GUI var mı? (Evet/Hayır/Bulamadım) + snippet\n"
            # "2) Nereden çalıştırılır? + snippet\n"
            # "3) Ana akış (3 madde)\n"
        )
    }
]

resp = llm.create_chat_completion(
    messages=messages,
    temperature=0.1,
    max_tokens=260
)

print(resp["choices"][0]["message"]["content"])


llama_init_from_model: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


1) MenuManager sınıfı, uygulamanın menü bar'ını yönetir.
2) MenuManager sınıfı, main_app.py dosyasında tanımlanmıştır.
3) Uygulamanın menü bar'ı, gui_components.py dosyasında yönetiliyor.


In [ ]:
!pip -q install streamlit chromadb sentence-transformers llama-cpp-python
!apt-get -qq update
!apt-get -qq install -y wget
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 144.1 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package cloudflared.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2025.11.1) ...
Setting up cloudflared (2025.11.1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!pkill -f streamlit || true
!pkill -f cloudflared || true
!lsof -i :8501 || true


^C
^C


In [ ]:
%%writefile app_ui.py
import json
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from llama_cpp import Llama
import streamlit as st

CHROMA_DIR = "outputs/context/vector_db"
COLLECTION = "repo_chunks"
EMBED_MODEL = "all-MiniLM-L6-v2"
LLM_PATH = "models/qwen2.5-coder-3b-instruct-q4_k_m.gguf"
TOP_K = 5

@st.cache_resource
def load_chroma():
    embed_fn = SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
    client = chromadb.PersistentClient(path=CHROMA_DIR)
    return client.get_or_create_collection(name=COLLECTION, embedding_function=embed_fn)

@st.cache_resource
def load_llm():
    return Llama(model_path=LLM_PATH, n_ctx=2048, n_threads=4, verbose=False)

def retrieve(col, query: str, top_k: int = TOP_K):
    res = col.query(query_texts=[query], n_results=top_k)
    evidence = []
    for i, doc in enumerate(res["documents"][0], start=1):
        meta = res["metadatas"][0][i-1]
        source = meta.get("source_path", "bilinmiyor")
        if source.endswith("_empty.py"):
            continue
        evidence.append({
            "id": f"E{i}",
            "source_path": source,
            "start_line": meta.get("start_line", "?"),
            "end_line": meta.get("end_line", "?"),
            "title": meta.get("title", ""),
            "quote": "\n".join(doc.splitlines()[:2]),
        })
    return evidence

def answer_with_llm(llm, query: str, evidence):
    ev_text = "\n".join([
        f"{e['id']} | {e['source_path']} | L{e['start_line']}-L{e['end_line']} | {e['title']}\n"
        f"ALINTI:\n{e['quote']}\n"
        for e in evidence
    ])

    messages = [
        {"role": "system", "content":
            "Türkçe yaz.\n"
            "SADECE aşağıdaki KANITLARA dayan.\n"
            "Kanıtlarda yoksa 'BULAMADIM' de.\n"
            "Dosya/satır uydurma. Sadece E1/E2 gibi kanıt kimliklerini seç.\n"
            "JSON dışında hiçbir şey yazma."
        },
        {"role": "user", "content":
            f"SORU: {query}\n\nKANITLAR:\n{ev_text}\n\n"
            "ŞU JSON'U üret:\n"
            "{\n"
            '  "answer": "2-5 cümlelik net cevap",\n'
            '  "use_evidence": ["E1","E2"]\n'
            "}"
        }
    ]

    resp = llm.create_chat_completion(messages=messages, temperature=0.1, max_tokens=350)
    raw = resp["choices"][0]["message"]["content"].strip()
    data = json.loads(raw)
    used = set(data.get("use_evidence", []))
    used_evs = [e for e in evidence if e["id"] in used]
    return data["answer"], used_evs

st.set_page_config(page_title="Dev-Pulse QA", layout="centered")
st.title("Code QA Chatbot System")
st.caption("Sorunu yaz → sistem kodlardan kanıtlı cevap versin.")

col = load_chroma()
llm = load_llm()

q = st.chat_input("Sorunu yaz (ör: 'MenuManager ne yapıyor?')")
if q:
    st.chat_message("user").markdown(q)
    with st.spinner("Kodlarda arıyorum..."):
        evidence = retrieve(col, q, TOP_K)
        if not evidence:
            ans = "BULAMADIM. (Kodlarda bu soruya dair kanıt yakalayamadım.)"
            used = []
        else:
            ans, used = answer_with_llm(llm, q, evidence)

    st.chat_message("assistant").markdown(ans)
    if used:
        st.markdown("**Kanıtlar**")
        for e in used:
            st.markdown(
                f"- `{e['source_path']}` | **L{e['start_line']}-L{e['end_line']}** | *{e['title']}*\n"
                f"  - Alıntı: `{e['quote']}`"
            )


Writing app_ui.py


In [ ]:
import subprocess, time, textwrap

# Streamlit arkaplanda başlasın
_ = subprocess.Popen(["streamlit", "run", "app_ui.py", "--server.port", "8501", "--server.address", "0.0.0.0"])

time.sleep(2)

# Cloudflare tunnel ile link al
!cloudflared tunnel --url http://localhost:8501 --no-autoupdate


2025-12-29T04:41:58Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2025-12-29T04:41:58Z INF Requesting new quick Tunnel on trycloudflare.com...
2025-12-29T04:42:01Z INF +--------------------------------------------------------------------------------------------+
2025-12-29T04:42:01Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2025-12-29T04:42:01Z INF |  https://fax-introductory-proposition-creative.tryclou

In [ ]:
import time, random, re, statistics

def pick_titles(col, n=30):
    res = col.query(query_texts=["class def function main init filter GUI"], n_results=50)
    titles = []
    metas = res["metadatas"][0]
    for m in metas:
        t = (m.get("title") or "").strip()
        sp = (m.get("source_path") or "").strip()
        if t and sp and not sp.endswith("_empty.py"):
            titles.append((t, sp))
    # uniq
    seen = set()
    out = []
    for t, sp in titles:
        key = (t, sp)
        if key not in seen:
            seen.add(key)
            out.append(key)
    random.shuffle(out)
    return out[:n]

def gen_question(title):
    # title örnekleri: "class MenuManager" / "def _setup_menu" / vs.
    m = title.strip()
    if m.startswith("class "):
        name = m.replace("class ", "").strip()
        return f"{name} sınıfı ne yapıyor?"
    if m.startswith("def "):
        name = m.replace("def ", "").strip()
        return f"{name} fonksiyonu ne iş yapıyor? Kısaca açıkla."
    return f"{m} nedir ve projede ne için kullanılıyor?"

def simple_relevance(q, doc):
    q_words = [w.lower() for w in re.findall(r"[a-zA-Z_]+", q) if len(w) >= 4]
    d = doc.lower()
    if not q_words:
        return 0.0
    hit = sum(1 for w in set(q_words) if w in d)
    return hit / len(set(q_words))

def run_auto_tests(col, llm=None, k=5, n_tests=10):
    pairs = pick_titles(col, n=40)
    if not pairs:
        print("Hiç title bulamadım. (DB boş olabilir ya da metadata yok.)")
        return

    tests = []
    for i in range(min(n_tests, len(pairs))):
        title, sp = pairs[i]
        q = gen_question(title)

        t0 = time.time()
        res = col.query(query_texts=[q], n_results=k)
        t_retr = time.time() - t0

        docs = res["documents"][0]
        metas = res["metadatas"][0]

        # metrikler
        rel_scores = [simple_relevance(q, d) for d in docs]
        precision_like = sum(1 for s in rel_scores if s >= 0.25) / k  # 0.25 eşik: pratik
        has_lines = sum(1 for m in metas if m.get("start_line") not in [None,"?"] and m.get("end_line") not in [None,"?"]) / k

        # LLM metriği
        t_llm = None
        used_evidence_count = None
        if llm is not None:
            ev_text = "\n".join([
                f"E{j+1} | {metas[j].get('source_path')} | L{metas[j].get('start_line')}-{metas[j].get('end_line')} | {metas[j].get('title')}\n"
                f"ALINTI: {docs[j].splitlines()[0] if docs[j].splitlines() else ''}"
                for j in range(len(docs))
            ])
            prompt = (
                f"SORU: {q}\n\nKANITLAR:\n{ev_text}\n\n"
                "JSON üret:\n"
                '{"answer":"kisa cevap","use_evidence":["E1","E2"]}'
            )
            t1 = time.time()
            out = llm.create_chat_completion(
                messages=[
                    {"role":"system","content":"Türkçe yaz. Sadece KANITLARA dayan. Sadece JSON."},
                    {"role":"user","content":prompt}
                ],
                temperature=0.1,
                max_tokens=220
            )
            t_llm = time.time() - t1
            txt = out["choices"][0]["message"]["content"].strip()
            try:
                import json
                data = json.loads(txt)
                used_evidence_count = len(data.get("use_evidence", []))
            except:
                used_evidence_count = 0

        tests.append({
            "q": q,
            "retr_s": t_retr,
            "prec_like": precision_like,
            "meta_ok": has_lines,
            "llm_s": t_llm,
            "used_ev": used_evidence_count
        })

        print(f"\n#{i+1} Q: {q}")
        print(f"  Retrieval: {t_retr:.2f}s | Precision-like@{k}: {precision_like:.2f} | MetaOK: {has_lines:.2f}")
        if llm is not None:
            print(f"  LLM: {t_llm:.2f}s | used_evidence: {used_evidence_count}")

    # özet
    print("\n=== ÖZET ===")
    print("Retrieval(s) avg:", round(statistics.mean(t["retr_s"] for t in tests), 2))
    print("Precision-like avg:", round(statistics.mean(t["prec_like"] for t in tests), 2))
    print("MetaOK avg:", round(statistics.mean(t["meta_ok"] for t in tests), 2))
    if llm is not None:
        llm_times = [t["llm_s"] for t in tests if t["llm_s"] is not None]
        print("LLM(s) avg:", round(statistics.mean(llm_times), 2))
        print("used_evidence avg:", round(statistics.mean(t["used_ev"] for t in tests), 2))

# ÇALIŞTIR:
run_auto_tests(col, llm=llm, k=5, n_tests=10)



#1 Q: BlurFilter sınıfı ne yapıyor?
  Retrieval: 0.03s | Precision-like@5: 0.60 | MetaOK: 1.00
  LLM: 42.20s | used_evidence: 2

#2 Q: __str__ fonksiyonu ne iş yapıyor? Kısaca açıkla.
  Retrieval: 0.02s | Precision-like@5: 0.20 | MetaOK: 1.00
  LLM: 50.40s | used_evidence: 1

#3 Q: process fonksiyonu ne iş yapıyor? Kısaca açıkla.
  Retrieval: 0.02s | Precision-like@5: 0.60 | MetaOK: 1.00
  LLM: 41.67s | used_evidence: 2

#4 Q: _setup_control_panel fonksiyonu ne iş yapıyor? Kısaca açıkla.
  Retrieval: 0.02s | Precision-like@5: 0.40 | MetaOK: 1.00
  LLM: 39.22s | used_evidence: 1

#5 Q: __init__ fonksiyonu ne iş yapıyor? Kısaca açıkla.
  Retrieval: 0.02s | Precision-like@5: 0.80 | MetaOK: 1.00
  LLM: 41.03s | used_evidence: 2

#6 Q: _create_filter_menu fonksiyonu ne iş yapıyor? Kısaca açıkla.
  Retrieval: 0.02s | Precision-like@5: 0.80 | MetaOK: 1.00
  LLM: 47.74s | used_evidence: 2

#7 Q: Filter sınıfı ne yapıyor?
  Retrieval: 0.02s | Precision-like@5: 0.80 | MetaOK: 1.00
  LLM: 42.05s

In [ ]:
import json, os
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from llama_cpp import Llama

# --- Ayarlar ---
CHROMA_DIR = "outputs/context/vector_db"
COLLECTION = "repo_chunks"
MODEL_PATH = "models/qwen2.5-coder-3b-instruct-q4_k_m.gguf"
OUT_README = "README_generated.md"

# 1) index.json'dan dosya listesi + requirements var mı?
with open("outputs/context/index.json", "r", encoding="utf-8") as f:
    index = json.load(f)

files = [x["path"] for x in index["files"]]
files_set = set(files)
has_requirements = any(p.lower().endswith("requirements.txt") for p in files)

file_tree = "\n".join([f"- {p}" for p in files])

# 2) Vector DB'den README için daha hedefli context çek
embed_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path=CHROMA_DIR)
col = client.get_collection(name=COLLECTION, embedding_function=embed_fn)

queries = [
    "main.py tk.Tk root.mainloop ImageProcessingApplication nasıl çalıştırılır",
    "setup_gui gui_components ttk Frame Canvas menü var mı",
    "pip install requirements.txt bağımlılıklar pillow opencv numpy import",
    "filters hog HOGDescriptor person detection trained_classifier.pkl var mı"
]

ctx_blocks = []
for q in queries:
    res = col.query(query_texts=[q], n_results=4)
    for i in range(4):
        meta = res["metadatas"][0][i]
        txt = res["documents"][0][i][:900]
        ctx_blocks.append(f"[Q:{q}]\nDosya: {meta.get('source_path')} | {meta.get('title')}\n{txt}")

# Context'i çok şişirmeyelim
context_text = "\n\n---\n\n".join(ctx_blocks[:12])

# 3) Qwen
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096,
    n_threads=4,
    verbose=False
)

# 4) Sıkı ve "uydurma engelleyici" prompt
messages = [
    {
        "role": "system",
        "content": (
            "Sen çok titiz bir teknik yazarsın.\n"
            "KURAL 1: SADECE verilen FILE_LIST ve CONTEXT'e dayan.\n"
            "KURAL 2: FILE_LIST'te olmayan dosyayı varmış gibi söyleme.\n"
            "KURAL 3: CONTEXT'te görmediğin özelliği uydurma (menü/kısayol vb.).\n"
            "KURAL 4: Emin değilsen 'Bilinmiyor' yaz.\n"
            "KURAL 5: 'git clone' ve hayali GitHub linki ASLA yazma.\n"
            "KURAL 6: Kurulumda requirements.txt ancak gerçekten varsa yaz.\n"
            "Türkçe, net ve teknik yaz."
        )
    },
    {
        "role": "user",
        "content": (
            f"FILE_LIST (projede gerçekten olan dosyalar):\n{file_tree}\n\n"
            f"requirements.txt var mı? -> {'EVET' if has_requirements else 'HAYIR'}\n\n"
            f"CONTEXT (projeden çekilmiş parçalar):\n{context_text}\n\n"
            "Şimdi bu projeye uygun, DETAYLI ama UYDURMASIZ bir README oluştur.\n"
            "Başlıklar:\n"
            "1) Proje Özeti\n"
            "2) Ne Yapar? (somut)\n"
            "3) Özellikler (sadece kanıtlı)\n"
            "4) Gereksinimler (yalnızca kanıtlı kütüphaneler)\n"
            "5) Kurulum (requirements varsa kullan, yoksa pip install satırlarını yaz)\n"
            "6) Çalıştırma (hangi dosya ve komut)\n"
            "7) GUI Kullanımı (yalnızca context'te gördüğün kadar)\n"
            "8) Proje Yapısı (dosyaları gruplandır: GUI / core / utils)\n"
            "9) Ana Akış (numaralı, main.py akışı)\n"
            "10) Bilinen Sınırlamalar (emin olmadıklarına 'Bilinmiyor' yaz)\n"
            "11) Geliştirme Fikirleri\n\n"
            "Komutlar kod bloğu içinde olsun."
        )
    }
]

resp = llm.create_chat_completion(
    messages=messages,
    temperature=0.1,
    max_tokens=1600
)

readme = resp["choices"][0]["message"]["content"].strip()

with open(OUT_README, "w", encoding="utf-8") as f:
    f.write(readme)

print(f"README üretildi: {OUT_README}")
print("\n--- ÖNİZLEME (ilk 60 satır) ---\n")
print("\n".join(readme.splitlines()[:60]))



llama_init_from_model: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


README üretildi: README_generated.md

--- ÖNİZLEME (ilk 60 satır) ---

# OOP Image Processing Application

## Proje Özeti
Bu proje, OpenCV ve Pillow kütüphanelerini kullanarak resim işleme uygulaması oluşturur. Uygulama, resimleri yükler, temizler, sepia matris transformationsu uygular ve farklı filtreler uygular. Projede, GUI (Grafik Kullanıcı Arayüzü) ve işlevlerin ayrı ayrı gruplandırılmıştır.

## Ne Yapar?
1. Resimleri yükler ve temizler.
2. Sepia matris transformationsu uygular.
3. Filtreler uygular (örneğin, kontrast artırma, eşikleme vb.).
4. Resimleri görsel olarak gösterir.

## Özellikler
1. Resim yükleme ve temizleme.
2. Sepia matris transformationsu.
3. Filtreler uygulama.
4. GUI (Grafik Kullanıcı Arayüzü) ve işlevlerin ayrı ayrı gruplandırılmıştır.

## Gereksinimler
1. Pillow
2. OpenCV
3. NumPy
4. Matplotlib
5. Scikit-learn
6. Scipy
7. Joblib
8. Imutils
9. Jupyter notebook (isteğe bağlı)

## Kurulum
Projede requirements.txt dosyası var, bu yüzden aşağıdaki komutu çalıştırar